In [1]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- NEW XGBOOST IMPORT ---
from xgboost import XGBRegressor

print("Starting Machine Learning Pipeline...")

Starting Machine Learning Pipeline...


In [2]:
print("Loading databases into memory...")

RESULTS_DIR= "data/processed"
# Read each CSV file into its own DataFrame
df1 = pd.read_csv(f"{RESULTS_DIR}/df1_baseline.csv")
df2 = pd.read_csv(f"{RESULTS_DIR}/df2_decisiontree.csv")
df3 = pd.read_csv(f"{RESULTS_DIR}/df3_regressiondata_removedoutliers.csv")
df4 = pd.read_csv(f"{RESULTS_DIR}/df4_weather_condition.csv")


print("All 4 databases loaded successfully!")

# Quick verification check
print(f"Database 1 Shape: {df1.shape}")
print(f"Database 2 Shape: {df2.shape}")
print(f"Database 3 Shape: {df3.shape}")
print(f"Database 4 Shape: {df4.shape}")

Loading databases into memory...
All 4 databases loaded successfully!
Database 1 Shape: (8465, 14)
Database 2 Shape: (8465, 13)
Database 3 Shape: (8313, 13)
Database 4 Shape: (8465, 13)


In [3]:
features_1 = df1.drop(columns=["Rented_Bike_Count"])
target_1 = df1["Rented_Bike_Count"]

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(features_1, target_1, test_size = 0.20, random_state=0)


In [4]:
features_2 = df2.drop(columns=["Rented_Bike_Count"])
target_2 = df2["Rented_Bike_Count"]

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(features_2, target_2, test_size = 0.20, random_state=0)


In [5]:
features_3 = df3.drop(columns=["Rented_Bike_Count"])
target_3 = df3["Rented_Bike_Count"]

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(features_3, target_3, test_size = 0.20, random_state=0)

In [6]:
scaler = StandardScaler()
X_train_norm_3 = scaler.fit_transform(X_train_3)
X_test_norm_3 = scaler.transform(X_test_3)
X_train_norm_3 = pd.DataFrame(X_train_norm_3,columns=X_train_3.columns)
X_test_norm_3 = pd.DataFrame(X_test_norm_3,columns=X_test_3.columns)

In [7]:
features_4 = df4.drop(columns=["Rented_Bike_Count"])
target_4 = df4["Rented_Bike_Count"]

X_train_4, X_test_4, y_train_4, y_test_4 = train_test_split(features_4, target_4, test_size = 0.20, random_state=0)

In [8]:
datasets = [

    # DATASET 1 (raw)
    {
        "name": "baseline",
        "X_train": X_train_1,
        "X_test": X_test_1,
        "y_train": y_train_1,
        "y_test": y_test_1
    },

    # DATASET 2
    {
        "name": "decision_tree",
        "X_train": X_train_2,
        "X_test": X_test_2,
        "y_train": y_train_2,
        "y_test": y_test_2
    },

    # DATASET 3 (normalized)
    {
        "name": "normalized",
        "X_train": X_train_norm_3,
        "X_test": X_test_norm_3,
        "y_train": y_train_3,
        "y_test": y_test_3
    },

    # DATASET 4
    {
        "name": "weather_condition",
        "X_train": X_train_4,
        "X_test": X_test_4,
        "y_train": y_train_4,
        "y_test": y_test_4
    }
] 

In [9]:
print(f"DB 1 Training shapes: X={X_train_1.shape}, y={y_train_1.shape}")
print(f"DB 2 Training shapes: X={X_train_2.shape}, y={y_train_2.shape}")
print(f"DB 3 Training shapes: X={X_train_3.shape}, y={y_train_3.shape}")
print(f"DB 4 Training shapes: X={X_train_4.shape}, y={y_train_4.shape}")

DB 1 Training shapes: X=(6772, 13), y=(6772,)
DB 2 Training shapes: X=(6772, 12), y=(6772,)
DB 3 Training shapes: X=(6650, 12), y=(6650,)
DB 4 Training shapes: X=(6772, 12), y=(6772,)


In [ ]:
grid = {"n_estimators": [50, 100, 200,500],
        "estimator__max_leaf_nodes": [250, 500, 1000, None],
        "estimator__max_depth":[10,30,50]}
model = GridSearchCV(estimator = ada_reg, param_grid = grid, cv=5)
model.fit(X_train_norm, y_train)
model.best_params_
best_model = model.best_estimator_

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = [
    {"name": "LinearRegression", "model": LinearRegression()},
    
    {"name": "RandomForest",
     "model": RandomForestRegressor(
         n_estimators=100,
         max_depth=20,
         random_state=42
     )},
    
    {"name": "XGBoost",
     "model": XGBRegressor(
         n_estimators=100,
         max_depth=6,
         learning_rate=0.1,
         random_state=42
     )}
]

In [11]:
results = []

for ds in datasets:
    for m in models:

        m["model"].fit(ds["X_train"], ds["y_train"])

        pred = m["model"].predict(ds["X_test"])

        results.append({
            "dataset": ds["name"],
            "model": m["name"],
            "MAE": mean_absolute_error(ds["y_test"], pred),
            "RMSE": np.sqrt(mean_squared_error(ds["y_test"], pred)),
            "R2": r2_score(ds["y_test"], pred),
        })

        print(f"{ds['name']:>12} | {m['name']:<16} done")

results_df = (
    pd.DataFrame(results)
      .sort_values("R2", ascending=False)
      .reset_index(drop=True)
)

results_df

    baseline | LinearRegression done
    baseline | RandomForest     done
    baseline | XGBoost          done
decision_tree | LinearRegression done
decision_tree | RandomForest     done
decision_tree | XGBoost          done
  normalized | LinearRegression done
  normalized | RandomForest     done
  normalized | XGBoost          done
weather_condition | LinearRegression done
weather_condition | RandomForest     done
weather_condition | XGBoost          done


,dataset,model,MAE,RMSE,R2
0,normalized,XGBoost,98.735847,158.698958,0.926798
1,normalized,RandomForest,97.530253,164.093924,0.921736
2,baseline,XGBoost,103.540962,177.363531,0.919959
3,baseline,RandomForest,107.357027,188.108377,0.909967
4,decision_tree,XGBoost,108.049393,191.399370,0.906789
5,decision_tree,RandomForest,112.964312,204.936558,0.893138
6,weather_condition,XGBoost,128.525024,217.338455,0.879813
7,weather_condition,RandomForest,137.283265,240.124374,0.853291
8,normalized,LinearRegression,297.202130,387.164894,0.564319
9,baseline,LinearRegression,321.572239,429.590404,0.530437


In [14]:
average_bikes = df1['Rented_Bike_Count'].mean()
print(f"Average bikes rented per hour: {average_bikes:.0f}")

# Let's say your XGBoost MAE was 150
error_percentage = (150 / average_bikes) * 100
print(f"Your error is only about {error_percentage:.1f}% of the average volume!")

Average bikes rented per hour: 729
Your error is only about 20.6% of the average volume!


In [12]:
# 1. Train a quick Decision Tree on Database 1
detective_tree = DecisionTreeRegressor(random_state=42)
detective_tree.fit(datasets[0]["X_train"], datasets[0]["y_train"])

# 2. Force it to confess which features it used!
importances = pd.Series(detective_tree.feature_importances_, index=datasets[0]["X_train"].columns)

print("🚨 FEATURE IMPORTANCE REPORT 🚨")
print("If any column is near 1.0, that is your leaked target!")
print("-" * 40)
print(importances.sort_values(ascending=False).head())

🚨 FEATURE IMPORTANCE REPORT 🚨
If any column is near 1.0, that is your leaked target!
----------------------------------------
Temperature        0.348458
Hour               0.321513
Solar_Radiation    0.100725
Humidity           0.099584
DayOfWeek          0.039902
dtype: float64
